# Demo 03 — Validation Report

This notebook demonstrates the **statistical validation** pipeline in SynthoHive. It shows how to:

1. Create real and synthetic datasets (with deliberate distribution shifts)
2. Generate an HTML validation report
3. Generate a JSON metrics file for programmatic analysis

The `ValidationReport` class compares marginal distributions, pairwise correlations, and computes summary quality scores.

In [ ]:
import numpy as np
import pandas as pd

from syntho_hive.validation.report_generator import ValidationReport

## 1. Create Real Data

We generate a 300-row dataset with mixed types: integer user IDs, ages, categorical region, continuous monthly spend, and a binary active flag.

In [ ]:
def make_real_data(num_rows: int = 300) -> pd.DataFrame:
    rng = np.random.default_rng(21)
    df = pd.DataFrame(
        {
            "user_id": np.arange(1, num_rows + 1),
            "age": rng.integers(18, 70, size=num_rows),
            "region": rng.choice(
                ["NE", "SE", "MW", "W"], size=num_rows, p=[0.25, 0.25, 0.25, 0.25]
            ),
            "monthly_spend": rng.normal(120, 35, size=num_rows).round(2),
            "active": rng.choice([0, 1], size=num_rows, p=[0.35, 0.65]),
        }
    )
    return df

real_df = make_real_data()
print(f"Real data shape: {real_df.shape}")
real_df.head()

## 2. Create Synthetic Variant with Distribution Shifts

We deliberately introduce shifts in `monthly_spend`, `active`, and `region` distributions so the validator surfaces measurable differences.

In [ ]:
def make_synthetic_variant(real_df: pd.DataFrame) -> pd.DataFrame:
    """Create a synthetic dataset with slight distribution shifts."""
    rng = np.random.default_rng(22)
    synth = real_df.copy()
    synth["monthly_spend"] = (
        synth["monthly_spend"] * rng.normal(1.05, 0.05, size=len(real_df))
    ).round(2)
    synth["active"] = rng.choice([0, 1], size=len(real_df), p=[0.4, 0.6])
    synth["region"] = rng.choice(
        ["NE", "SE", "MW", "W"], size=len(real_df), p=[0.35, 0.2, 0.25, 0.2]
    )
    return synth

synthetic_df = make_synthetic_variant(real_df)
print(f"Synthetic data shape: {synthetic_df.shape}")
synthetic_df.head()

## 3. Generate HTML Validation Report

The `ValidationReport.generate()` method accepts dictionaries mapping table names to DataFrames. When `output_path` ends in `.html`, it writes a self-contained HTML report.

In [ ]:
report = ValidationReport()

report.generate(
    real_data={"users": real_df},
    synth_data={"users": synthetic_df},
    output_path="validation_report.html",
)
print("Wrote HTML report to validation_report.html")

## 4. Generate JSON Metrics

When `output_path` ends in `.json`, the same `generate()` method writes structured metrics suitable for programmatic analysis or CI pipelines.

In [ ]:
report.generate(
    real_data={"users": real_df},
    synth_data={"users": synthetic_df},
    output_path="validation_metrics.json",
)
print("Wrote JSON metrics to validation_metrics.json")

## 5. Quick Statistical Comparison

A manual comparison of summary statistics to see the distribution shifts.

In [ ]:
print("=== Real Data ===")
print(real_df.describe())
print("\n=== Synthetic Data ===")
print(synthetic_df.describe())